# Spark

In [ ]:
# 1. Cài Spark cho phiên Colab hiện tại
%pip -q install pyspark==4.0.5

from pyspark.sql import SparkSession, functions as F
from IPython.display import display
import pandas as pd

# 2. Khởi động Spark
spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("HMDA_Denial_Diagnostics")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

# 3. Đọc dữ liệu do Spark làm sạch
# Không dùng bảng Pandas để tạo dữ liệu Spark
s2_diag_sp = (
    spark.read.parquet(str(S2_SPARK_PATH))
    .select(*DIAG_COLS)
)

# 4. Chuẩn hóa bốn cột mã lý do thành chuỗi
for col in REASON_COLS:
    trimmed = F.trim(F.col(col).cast("string"))

    s2_diag_sp = s2_diag_sp.withColumn(
        col,
        F.when(trimmed == "", F.lit(None).cast("string"))
        .otherwise(trimmed),
    )

# Gộp bốn cột thành một danh sách mã trên từng hồ sơ
s2_diag_sp = s2_diag_sp.withColumn(
    "_reported_reasons",
    F.filter(
        F.array(*[F.col(c) for c in REASON_COLS]),
        lambda x: x.isNotNull(),
    ),
)

# Danh sách lý do hợp lệ, mỗi mã chỉ giữ một lần trong mỗi hồ sơ
s2_diag_sp = s2_diag_sp.withColumn(
    "reason_codes",
    F.array_distinct(
        F.filter(
            F.col("_reported_reasons"),
            lambda x: x.isin(*list(REASON_LABELS)),
        )
    ),
)

# 5. Kiểm tra toàn bộ dữ liệu trong một lần tổng hợp
checks_sp = s2_diag_sp.agg(
    F.count("*").alias("total_records"),

    F.sum(
        F.when(
            F.coalesce(
                F.col("action_taken").cast("string") == "3",
                F.lit(False),
            ),
            0,
        ).otherwise(1)
    ).alias("invalid_action_records"),

    F.sum(
        F.when(F.size("reason_codes") > 0, 1).otherwise(0)
    ).alias("with_valid_reason"),

    F.sum(
        F.when(F.size("reason_codes") == 0, 1).otherwise(0)
    ).alias("without_valid_reason"),

    F.sum(
        F.size(
            F.filter(
                F.col("_reported_reasons"),
                lambda x: ~x.isin(*list(REASON_LABELS)),
            )
        )
    ).alias("unexpected_code_slots"),

    F.sum(
        F.size("_reported_reasons")
        - F.size(F.array_distinct("_reported_reasons"))
    ).alias("duplicate_reason_slots"),
).first().asDict()

# 6. Đối chiếu với kết quả Pandas
input_comparison_pd = pd.DataFrame([
    {
        "metric": "Total denied applications",
        "pandas": n_s2,
        "spark": checks_sp["total_records"],
    },
    {
        "metric": "Applications with valid reason",
        "pandas": n_with_reason,
        "spark": checks_sp["with_valid_reason"],
    },
    {
        "metric": "Applications without valid reason",
        "pandas": n_without_reason,
        "spark": checks_sp["without_valid_reason"],
    },
    {
        "metric": "Duplicate reason slots",
        "pandas": duplicate_reason_slots,
        "spark": checks_sp["duplicate_reason_slots"],
    },
])

input_comparison_pd["match"] = (
    input_comparison_pd["pandas"]
    == input_comparison_pd["spark"]
)

print("\nKIỂM TRA ĐẦU VÀO PANDAS – SPARK")
display(input_comparison_pd)

print(
    "Hồ sơ không có action_taken = 3:",
    checks_sp["invalid_action_records"],
)
print(
    "Số ô có mã ngoài danh sách 1–9:",
    checks_sp["unexpected_code_slots"],
)

# 7. Lưu kết quả kiểm tra
SPARK_DIAG_OUT = DIAG_OUT / "spark"
CORRECTNESS_DIAG_OUT = DIAG_OUT / "correctness"

SPARK_DIAG_OUT.mkdir(parents=True, exist_ok=True)
CORRECTNESS_DIAG_OUT.mkdir(parents=True, exist_ok=True)

input_comparison_pd.to_csv(
    CORRECTNESS_DIAG_OUT / "01_input_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)

assert checks_sp["invalid_action_records"] == 0, \
    "Có hồ sơ không thuộc nhóm bị từ chối."
assert checks_sp["unexpected_code_slots"] == 0, \
    "Có mã lý do cần kiểm tra thêm."
assert input_comparison_pd["match"].all(), \
    "Đầu vào Spark chưa khớp Pandas."

print("\nPASS — đầu vào Spark khớp Pandas, sẵn sàng đối chiếu phân tích.")

Spark version: 4.0.5

KIỂM TRA ĐẦU VÀO PANDAS – SPARK


,metric,pandas,spark,match
0,Total denied applications,2103462,2103462,True
1,Applications with valid reason,2074770,2074770,True
2,Applications without valid reason,28692,28692,True
3,Duplicate reason slots,0,0,True


Hồ sơ không có action_taken = 3: 0
Số ô có mã ngoài danh sách 1–9: 0

PASS — đầu vào Spark khớp Pandas, sẵn sàng đối chiếu phân tích.


In [ ]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from IPython.display import display

# 1. Mỗi hồ sơ có một danh sách lý do không trùng mã.
# explode tạo một dòng cho mỗi lý do của hồ sơ.
# Hồ sơ có nhiều lý do sẽ xuất hiện trong nhiều nhóm.
reason_counts_sp = (
    s2_diag_sp
    .select(F.explode("reason_codes").alias("reason_code"))
    .groupBy("reason_code")
    .agg(F.count("*").alias("applications"))
)

# Chỉ chuyển bảng tổng hợp nhỏ về Pandas để hiển thị/đối chiếu
spark_reason_counts_pd = reason_counts_sp.toPandas()

# 2. Bổ sung đủ chín mã, kể cả nếu một mã không xuất hiện
reason_frequency_spark_pd = pd.DataFrame({
    "reason_code": list(REASON_LABELS)
})

reason_frequency_spark_pd = reason_frequency_spark_pd.merge(
    spark_reason_counts_pd,
    on="reason_code",
    how="left",
    validate="one_to_one",
)

reason_frequency_spark_pd["applications"] = (
    reason_frequency_spark_pd["applications"]
    .fillna(0)
    .astype("int64")
)

reason_frequency_spark_pd["reason"] = (
    reason_frequency_spark_pd["reason_code"].map(REASON_LABELS)
)

# Mẫu số lấy từ kết quả kiểm tra đầu vào Spark
spark_total_denied = int(checks_sp["total_records"])
spark_with_reason = int(checks_sp["with_valid_reason"])

reason_frequency_spark_pd["share_of_all_denied_pct"] = (
    reason_frequency_spark_pd["applications"]
    / spark_total_denied * 100
)

reason_frequency_spark_pd["share_of_reason_reported_pct"] = (
    reason_frequency_spark_pd["applications"]
    / spark_with_reason * 100
)

reason_frequency_spark_pd = (
    reason_frequency_spark_pd[
        [
            "reason_code",
            "reason",
            "applications",
            "share_of_all_denied_pct",
            "share_of_reason_reported_pct",
        ]
    ]
    .sort_values(
        ["applications", "reason_code"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

print("BẢNG TẦN SUẤT LÝ DO — SPARK")
display(reason_frequency_spark_pd.round(2))

# 3. Ghép kết quả theo mã lý do, không theo thứ tự dòng
metrics = [
    "applications",
    "share_of_all_denied_pct",
    "share_of_reason_reported_pct",
]

reason_frequency_check_pd = (
    reason_frequency_pd[["reason_code"] + metrics]
    .merge(
        reason_frequency_spark_pd[["reason_code"] + metrics],
        on="reason_code",
        how="outer",
        suffixes=("_pandas", "_spark"),
        validate="one_to_one",
        indicator=True,
    )
)

assert reason_frequency_check_pd["_merge"].eq("both").all(), \
    "Hai bảng không có cùng danh sách mã lý do."

# Số hồ sơ phải khớp hoàn toàn
reason_frequency_check_pd["applications_match"] = (
    reason_frequency_check_pd["applications_pandas"]
    == reason_frequency_check_pd["applications_spark"]
)

# Phần trăm cho phép sai số làm tròn số thực rất nhỏ
for metric in metrics[1:]:
    reason_frequency_check_pd[f"{metric}_abs_diff"] = (
        reason_frequency_check_pd[f"{metric}_pandas"]
        - reason_frequency_check_pd[f"{metric}_spark"]
    ).abs()

    reason_frequency_check_pd[f"{metric}_match"] = np.isclose(
        reason_frequency_check_pd[f"{metric}_pandas"],
        reason_frequency_check_pd[f"{metric}_spark"],
        atol=1e-9,
        rtol=0,
    )

match_cols = [
    "applications_match",
    "share_of_all_denied_pct_match",
    "share_of_reason_reported_pct_match",
]

reason_frequency_check_pd["all_match"] = (
    reason_frequency_check_pd[match_cols].all(axis=1)
)

print("\nĐỐI CHIẾU PANDAS – SPARK")
display(
    reason_frequency_check_pd[
        [
            "reason_code",
            "applications_pandas",
            "applications_spark",
            *match_cols,
            "all_match",
        ]
    ]
)

# 4. Lưu bảng kết quả và bằng chứng đối chiếu
reason_frequency_spark_pd.to_csv(
    SPARK_DIAG_OUT / "04_reason_frequency.csv",
    index=False,
    encoding="utf-8-sig",
)

reason_frequency_check_pd.drop(columns="_merge").to_csv(
    CORRECTNESS_DIAG_OUT / "02_reason_frequency_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)

assert reason_frequency_check_pd["all_match"].all(), \
    "Có kết quả chưa khớp; gửi bảng đối chiếu để kiểm tra."

print("\nPASS — tần suất và tỷ trọng của cả 9 lý do khớp Pandas.")

BẢNG TẦN SUẤT LÝ DO — SPARK


,reason_code,reason,applications,share_of_all_denied_pct,share_of_reason_reported_pct
0,1,Debt-to-income ratio,773844,36.79,37.30
1,3,Credit history,679572,32.31,32.75
2,4,Collateral,347100,16.50,16.73
3,9,Other,276173,13.13,13.31
4,7,Credit application incomplete,269600,12.82,12.99
5,6,Unverifiable information,123489,5.87,5.95
6,5,"Insufficient cash (downpayment, closing costs)",118663,5.64,5.72
7,2,Employment history,42827,2.04,2.06
8,8,Mortgage insurance denied,1148,0.05,0.06



ĐỐI CHIẾU PANDAS – SPARK


,reason_code,applications_pandas,applications_spark,applications_match,share_of_all_denied_pct_match,share_of_reason_reported_pct_match,all_match
0,1,773844,773844,True,True,True,True
1,2,42827,42827,True,True,True,True
2,3,679572,679572,True,True,True,True
3,4,347100,347100,True,True,True,True
4,5,118663,118663,True,True,True,True
5,6,123489,123489,True,True,True,True
6,7,269600,269600,True,True,True,True
7,8,1148,1148,True,True,True,True
8,9,276173,276173,True,True,True,True



PASS — tần suất và tỷ trọng của cả 9 lý do khớp Pandas.


In [ ]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from IPython.display import display

# 1. Tạo các nhóm phân tích ngay trong Spark
# Mỗi hồ sơ thuộc:
# - nhóm ALL;
# - các nhóm lý do đã ghi;
# - hoặc NO_REASON nếu không có lý do hợp lệ.
group_codes_expr = F.concat(
    F.array(F.lit("ALL")),
    F.when(
        F.size("reason_codes") > 0,
        F.col("reason_codes"),
    ).otherwise(F.array(F.lit("NO_REASON"))),
)

financial_groups_sp = s2_diag_sp.select(
    F.explode(group_codes_expr).alias("group_code"),
    *[F.col(c) for c in FINANCIAL_VARS],
)

group_labels = {
    "ALL": "All denied applications",
    **REASON_LABELS,
    "NO_REASON": "No valid reason in cleaned data",
}

# 2. Tính từng biến để tránh gom quá nhiều dữ liệu cùng lúc
spark_summary_rows = []

for col, (variable, unit) in FINANCIAL_VARS.items():
    print(f"Đang tính: {variable}...", flush=True)

    numeric_value = F.col(col).cast("double")

    # Spark phân biệt NaN và null.
    # Chuyển NaN thành null để loại giá trị thiếu giống Pandas.
    values_sp = financial_groups_sp.select(
        "group_code",
        F.when(
            numeric_value.isNull() | F.isnan(numeric_value),
            F.lit(None).cast("double"),
        ).otherwise(numeric_value).alias("value"),
    )

    # percentile là phân vị chính xác, không dùng percentile_approx
    stats_rows = (
        values_sp
        .groupBy("group_code")
        .agg(
            F.count("*").alias("total_records"),
            F.count("value").alias("valid_records"),
            F.expr(
                "percentile(value, array(0.25D, 0.50D, 0.75D))"
            ).alias("quartiles"),
        )
        .collect()
    )

    # Chỉ đưa bảng tổng hợp nhỏ về Python
    for row in stats_rows:
        total = int(row["total_records"])
        valid = int(row["valid_records"])
        quartiles = row["quartiles"]

        q1, median, q3 = (
            quartiles if quartiles is not None
            else [np.nan, np.nan, np.nan]
        )

        spark_summary_rows.append({
            "group_code": row["group_code"],
            "group": group_labels[row["group_code"]],
            "variable": variable,
            "unit": unit,
            "total_records": total,
            "valid_records": valid,
            "missing_pct": (total - valid) / total * 100,
            "Q1": q1,
            "median": median,
            "Q3": q3,
            "IQR": q3 - q1,
        })

reason_financial_summary_spark_pd = pd.DataFrame(
    spark_summary_rows
)

# 3. Ghép kết quả theo nhóm và biến
keys = ["group_code", "variable"]
metrics = [
    "total_records",
    "valid_records",
    "missing_pct",
    "Q1",
    "median",
    "Q3",
    "IQR",
]

financial_check_pd = (
    reason_financial_summary_pd[keys + metrics]
    .merge(
        reason_financial_summary_spark_pd[keys + metrics],
        on=keys,
        how="outer",
        suffixes=("_pandas", "_spark"),
        validate="one_to_one",
        indicator=True,
    )
)

assert financial_check_pd["_merge"].eq("both").all(), \
    "Hai bảng chưa có cùng các nhóm và biến."

# 4. Kiểm tra từng chỉ tiêu
check_summary_rows = []

for metric in metrics:
    pandas_values = financial_check_pd[f"{metric}_pandas"]
    spark_values = financial_check_pd[f"{metric}_spark"]

    # Số lượng phải bằng nhau hoàn toàn.
    # Các chỉ tiêu số thực cho phép sai số rất nhỏ.
    tolerance = (
        0.0 if metric in ["total_records", "valid_records"]
        else 1e-6
    )

    matches = np.isclose(
        pandas_values.to_numpy(dtype=float),
        spark_values.to_numpy(dtype=float),
        atol=tolerance,
        rtol=0,
        equal_nan=True,
    )

    financial_check_pd[f"{metric}_match"] = matches

    differences = (pandas_values - spark_values).abs()

    check_summary_rows.append({
        "metric": metric,
        "rows_checked": len(financial_check_pd),
        "max_abs_difference": differences.max(),
        "tolerance": tolerance,
        "match": bool(matches.all()),
    })

financial_check_summary_pd = pd.DataFrame(check_summary_rows)

print("\nĐỐI CHIẾU THỐNG KÊ TÀI CHÍNH PANDAS – SPARK")
display(financial_check_summary_pd)

# 5. Lưu kết quả và bảng kiểm tra chi tiết
reason_financial_summary_spark_pd.to_csv(
    SPARK_DIAG_OUT / "06_financial_summary_by_reason.csv",
    index=False,
    encoding="utf-8-sig",
)

financial_check_pd.drop(columns="_merge").to_csv(
    CORRECTNESS_DIAG_OUT / "03_financial_comparison_detail.csv",
    index=False,
    encoding="utf-8-sig",
)

financial_check_summary_pd.to_csv(
    CORRECTNESS_DIAG_OUT / "04_financial_comparison_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

# Nếu có sai lệch, hiện dòng cần kiểm tra
match_cols = [f"{metric}_match" for metric in metrics]
failed_rows = financial_check_pd.loc[
    ~financial_check_pd[match_cols].all(axis=1)
]

if not failed_rows.empty:
    print("\nCÁC DÒNG CHƯA KHỚP:")
    display(failed_rows)

assert financial_check_summary_pd["match"].all(), \
    "Có thống kê chưa khớp; gửi bảng kết quả để kiểm tra."

print("\nPASS — thống kê tài chính theo lý do khớp Pandas.")

Đang tính: Income...
Đang tính: Loan amount...
Đang tính: CLTV...
Đang tính: Property value...

ĐỐI CHIẾU THỐNG KÊ TÀI CHÍNH PANDAS – SPARK


,metric,rows_checked,max_abs_difference,tolerance,match
0,total_records,44,0.0,0.000000,True
1,valid_records,44,0.0,0.000000,True
2,missing_pct,44,0.0,0.000001,True
3,Q1,44,0.0,0.000001,True
4,median,44,0.0,0.000001,True
5,Q3,44,0.0,0.000001,True
6,IQR,44,0.0,0.000001,True



PASS — thống kê tài chính theo lý do khớp Pandas.


In [ ]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from IPython.display import display

# 1. Chuẩn hóa nhãn DTI: giữ giá trị thiếu thành nhóm Missing
dti_text = F.trim(F.col("dti_band").cast("string"))

dti_groups_sp = (
    s2_diag_sp
    .select(
        F.explode(group_codes_expr).alias("group_code"),
        F.when(
            dti_text.isNull() | (dti_text == ""),
            F.lit("Missing"),
        ).otherwise(dti_text).alias("dti_band"),
    )
)

# 2. Đếm hồ sơ theo nhóm lý do và DTI hoàn toàn bằng Spark
dti_counts_spark_pd = (
    dti_groups_sp
    .groupBy("group_code", "dti_band")
    .agg(F.count("*").alias("applications"))
    .toPandas()
)

unexpected_bands = (
    set(dti_counts_spark_pd["dti_band"]) - set(DTI_ORDER)
)

assert not unexpected_bands, \
    f"Có nhãn DTI cần kiểm tra: {unexpected_bands}"

# 3. Bổ sung cả tổ hợp không có hồ sơ, với số lượng bằng 0
full_grid = pd.MultiIndex.from_product(
    [list(group_labels), DTI_ORDER],
    names=["group_code", "dti_band"],
).to_frame(index=False)

reason_dti_distribution_spark_pd = full_grid.merge(
    dti_counts_spark_pd,
    on=["group_code", "dti_band"],
    how="left",
    validate="one_to_one",
)

reason_dti_distribution_spark_pd["applications"] = (
    reason_dti_distribution_spark_pd["applications"]
    .fillna(0)
    .astype("int64")
)

reason_dti_distribution_spark_pd["group"] = (
    reason_dti_distribution_spark_pd["group_code"]
    .map(group_labels)
)

# Mẫu số gồm mọi hồ sơ trong nhóm, kể cả thiếu DTI
reason_dti_distribution_spark_pd["group_total"] = (
    reason_dti_distribution_spark_pd
    .groupby("group_code")["applications"]
    .transform("sum")
)

reason_dti_distribution_spark_pd["share_within_group_pct"] = (
    reason_dti_distribution_spark_pd["applications"]
    / reason_dti_distribution_spark_pd["group_total"]
    * 100
)

# 4. Đối chiếu theo nhóm lý do và DTI
keys = ["group_code", "dti_band"]
metrics = [
    "applications",
    "group_total",
    "share_within_group_pct",
]

dti_check_pd = (
    reason_dti_distribution_pd[keys + metrics]
    .merge(
        reason_dti_distribution_spark_pd[keys + metrics],
        on=keys,
        how="outer",
        suffixes=("_pandas", "_spark"),
        validate="one_to_one",
        indicator=True,
    )
)

assert dti_check_pd["_merge"].eq("both").all(), \
    "Hai bảng chưa có cùng các nhóm lý do và DTI."

dti_check_summary_rows = []

for metric in metrics:
    pandas_values = dti_check_pd[f"{metric}_pandas"]
    spark_values = dti_check_pd[f"{metric}_spark"]

    tolerance = (
        1e-9 if metric == "share_within_group_pct"
        else 0.0
    )

    matches = np.isclose(
        pandas_values.to_numpy(dtype=float),
        spark_values.to_numpy(dtype=float),
        atol=tolerance,
        rtol=0,
        equal_nan=True,
    )

    dti_check_pd[f"{metric}_match"] = matches

    dti_check_summary_rows.append({
        "metric": metric,
        "rows_checked": len(dti_check_pd),
        "max_abs_difference": (
            pandas_values - spark_values
        ).abs().max(),
        "tolerance": tolerance,
        "match": bool(matches.all()),
    })

dti_check_summary_pd = pd.DataFrame(dti_check_summary_rows)

print("ĐỐI CHIẾU PHÂN BỐ DTI PANDAS – SPARK")
display(dti_check_summary_pd)

# 5. Lưu kết quả và bằng chứng đối chiếu
reason_dti_distribution_spark_pd.to_csv(
    SPARK_DIAG_OUT / "09_dti_distribution_by_reason.csv",
    index=False,
    encoding="utf-8-sig",
)

dti_check_pd.drop(columns="_merge").to_csv(
    CORRECTNESS_DIAG_OUT / "05_dti_comparison_detail.csv",
    index=False,
    encoding="utf-8-sig",
)

dti_check_summary_pd.to_csv(
    CORRECTNESS_DIAG_OUT / "06_dti_comparison_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

match_cols = [f"{metric}_match" for metric in metrics]
failed_rows = dti_check_pd.loc[
    ~dti_check_pd[match_cols].all(axis=1)
]

if not failed_rows.empty:
    print("\nCÁC DÒNG CHƯA KHỚP:")
    display(failed_rows)

assert dti_check_summary_pd["match"].all(), \
    "Có kết quả chưa khớp; gửi bảng để kiểm tra."

print("\nPASS — phân bố DTI theo lý do khớp Pandas.")
print(
    "Đã hoàn tất đối chiếu: đầu vào, tần suất lý do, "
    "thống kê tài chính và phân bố DTI."
)

ĐỐI CHIẾU PHÂN BỐ DTI PANDAS – SPARK


,metric,rows_checked,max_abs_difference,tolerance,match
0,applications,88,0.0,0.000000e+00,True
1,group_total,88,0.0,0.000000e+00,True
2,share_within_group_pct,88,0.0,1.000000e-09,True



PASS — phân bố DTI theo lý do khớp Pandas.
Đã hoàn tất đối chiếu: đầu vào, tần suất lý do, thống kê tài chính và phân bố DTI.
